In [45]:
from pyspark.sql.functions import input_file_name

books_df = spark.read.text(
    "/user/input/Gutenberg/txt/*.txt"
)

books_df = books_df.withColumn("file_name", input_file_name())
books_df = books_df.withColumnRenamed("value", "text")

books_df.show(5, truncate=False)

+----------------------+---------------------------------------------------------------------------------------------+
|text                  |file_name                                                                                    |
+----------------------+---------------------------------------------------------------------------------------------+
|                      |hdfs://127.0.0.1:9000/user/input/Gutenberg/txt/James%20Fenimore%20Cooper___Oak%20Openings.txt|
|OAK OPENINGS          |hdfs://127.0.0.1:9000/user/input/Gutenberg/txt/James%20Fenimore%20Cooper___Oak%20Openings.txt|
|                      |hdfs://127.0.0.1:9000/user/input/Gutenberg/txt/James%20Fenimore%20Cooper___Oak%20Openings.txt|
|JAMES FENNIMORE COOPER|hdfs://127.0.0.1:9000/user/input/Gutenberg/txt/James%20Fenimore%20Cooper___Oak%20Openings.txt|
|                      |hdfs://127.0.0.1:9000/user/input/Gutenberg/txt/James%20Fenimore%20Cooper___Oak%20Openings.txt|
+----------------------+------------------------

In [ ]:
books_df = spark.read.text(
    "/user/input/Gutenberg/txt/*.txt",
    wholetext=True
)

In [ ]:
books_df = books_df.withColumn(
    "title",
    F.trim(F.split("text", "\n").getItem(1))
)

books_df = books_df.withColumn(
    "author",
    F.trim(F.split("text", "\n").getItem(3))
)
books_df.select("title", "author").show(5, truncate=False)

books_df = books_df.withColumn(
    "clean_text",
    F.lower(F.regexp_replace("text", "[^a-zA-Z ]", " "))
)

In [ ]:

#### 11. TF-IDF and Book Similarity

from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="clean_text",
    outputCol="words"
)

words_df = tokenizer.transform(books_df)

from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words"
)

filtered_df = remover.transform(words_df)
filtered_df.select("title", "filtered_words").show(2, truncate=False)
filtered_df.select("title", "filtered_words").show(2, truncate=False)

from pyspark.ml.feature import HashingTF

hashingTF = HashingTF(
    inputCol="filtered_words",
    outputCol="tf",
    numFeatures=10000
)

tf_df = hashingTF.transform(filtered_df)

from pyspark.ml.feature import IDF

idf = IDF(
    inputCol="tf",
    outputCol="tfidf"
)

idf_model = idf.fit(tf_df)
tfidf_df = idf_model.transform(tf_df)

tfidf_df.select("title", "tfidf").show(5, truncate=False)

small_df = tfidf_df.select("title", "tfidf").limit(400).toPandas()

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

vectors = np.array([row.toArray() for row in small_df["tfidf"]])

similarity_matrix = cosine_similarity(vectors)
similarity_df = pd.DataFrame(similarity_matrix)
similarity_df.to_csv("cosine_similarity_matrix.csv", index=False)

print(similarity_matrix)


In [ ]:
books_df.select(F.avg(F.length("title"))).show()

books_df.groupBy("author").count().orderBy(F.desc("count")).show(10)


In [ ]:
#####  12. Author Influence Network

## Author and release year extraction
books_df = books_df.withColumn(
    "header",
    F.substring("text", 1, 1000)
)
books_df = books_df.withColumn(
    "author",
    F.regexp_extract(
        "header",
        r"(?i)\bby\s+([^\r\n]+)",
        1
    )
)
books_df = books_df.withColumn(
    "author",
    F.trim(F.regexp_replace("author", r"\.$", ""))
)
books_df = books_df.filter(
    (~F.col("author").rlike("(?i)group")) &
    (~F.col("author").rlike("\\(")) &
    (~F.col("author").rlike("\\)")) &
    (F.col("author").rlike("^[A-Za-z\\.\\s]+$"))
)

#### Analysis

influence_df = books_df.alias("a").join(
    books_df.alias("b"),
    (F.col("a.author") != F.col("b.author")) &
    (F.col("a.release_year") < F.col("b.release_year")) &
    (F.col("b.release_year") - F.col("a.release_year") <= X)
).select(
    F.col("a.author").alias("author1"),
    F.col("b.author").alias("author2")
).distinct()

influence_df.show(10, truncate=False)
influence_df.count()

out_degree = influence_df.groupBy("author1").count().orderBy(F.desc("count"))
in_degree = influence_df.groupBy("author2").count().orderBy(F.desc("count"))

out_degree.show(5)
in_degree.show(5)
